> **Repository execution note.** Start JupyterLab from the **repository root** and place authorised source data in that same root directory (see [README](../README.md) and [Data requirements](../data/README.md)). These notebooks were originally distributed as standalone packages: any older instructions referring to an included ZIP, a `test/` folder, or bundled data do not apply to this GitHub repository. The notebooks preserve their original input schemas and experiment configurations. Results cached in the original files are intentionally excluded from this public-ready copy.

# Labour Productivity in Independent Hotels | Study for *Knowledge-Based Systems*

**Unified, annotated notebook ready to be adapted to the actual dataset.**  
Working version: October 29, 2026. Researcher: Luís Silvano and team.

## What is this notebook for?

This notebook brings **everything into one place**: data loading, filtering of independent hotels and years, preprocessing, model training, result comparison, and table export.

**Central research question:** Given information on a hotel's characteristics and location, can we estimate its **labour productivity**, measured by `f_labprodct`?

> **Important:** The original Excel dataset has not yet been provided. This file is therefore a reproducible program, but it **does not report actual results** until it is run with the project's dataset. No illustrative metric should be reported in the paper.


## Five-step user guide

1. Place this notebook in the same folder as the original Excel file: `2018- 2024 Luis Silvano - Doctoral Thesis Dataset.xlsx` (or change `FICHEIRO_DADOS` in the configuration).
2. Install the basic packages, open the notebook in **JupyterLab** or **VS Code**, and select the correct Python environment.
3. Start with `PROTOCOLO = "temporal"` and the core models. Choose **Run All**.
4. Review the `Resultados` table and open the CSV files generated in `resultados_kbs/`.
5. To run a different test, set `PROTOCOLO = "hoteis_novos"` and rerun the cells starting from the configuration.

### Installation

First, run the following in a terminal:

```bash
pip install pandas numpy scipy scikit-learn matplotlib openpyxl jupyter nbformat
```

For strong classical baselines:

```bash
pip install xgboost lightgbm catboost
```

**Only if you want to experiment with recent architectures** (preferably in a separate environment with an appropriate PyTorch installation):

```bash
pip install "pytabkit[models]" pytorch-tabnet tabicl tabpfn causilo interpret
```

*Notes:* PyTabKit/TabR may require a separate FAISS installation; some foundation models download pretrained weights and may require permissions/licences, memory, and a GPU. You do not need to install everything for the initial analysis. Optional libraries are imported only when their corresponding models are selected.


## Machine Learning Concepts, Without Excessive Jargon

| Term | Meaning in this project |
|---|---|
| **An observation** | One hotel in a particular year. The same hotel may appear in multiple years. |
| **Target variable (`y`)** | The outcome to explain/predict: `f_labprodct`, a numerical quantity. We therefore perform **regression**, not classification. |
| **Explanatory variables (`X`)** | Hotel and surrounding-area characteristics used to generate a prediction. |
| **Training** | The model learns patterns from the years or hotels assigned to the training set. |
| **Validation** | Comparison of models/configurations **without using the final test set**. |
| **Testing** | Final evaluation on observations not used to fit the models. |
| **Data leakage** | When a step inadvertently uses information that would not be available at prediction time. |
| **MAE / RMSE** | Error metrics: **lower is better**. RMSE penalizes large errors more heavily. |
| **R²** | Compares the model with a mean-based reference; it can be negative on the test set. **Higher is better**. |
| **Seed** | A number used to make randomized experiments reproducible. |

**Methodological warning:** This notebook replaces the previous classification preparation (`0/1`, class balancing, `f_classif`) with a regression-oriented approach. Class balancing is not appropriate when the target is continuous.


---
## 1. Configuration: Change Options Here

**First-time users:** Do not change anything except the Excel file name/path, if necessary.  
**Where are the years filtered?** In `ANO_INICIAL`, `ANO_FINAL`, `ANOS_TREINO`, `ANO_VALIDACAO`, `ANO_TESTE`, and the explicit filter in Section 3.

**Two distinct evaluation protocols** (do not conflate their interpretations):

- `temporal`: train on 2018–2022, validate on 2023, and test on 2024. The **same hotel may appear** across sets in different years. This assesses generalization to a subsequent year.
- `hoteis_novos`: hotels in the training, validation, and test sets **do not overlap**. Observations from 2018–2024 are used across all sets. This evaluates generalization across hotels, **not a 2024 forecast based on earlier information**.


In [ ]:
from pathlib import Path
import importlib.util
import importlib.metadata
import json
import platform
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.model_selection import GroupShuffleSplit
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.feature_selection import f_regression

SEED = 42

# 1) Change only if the Excel file has a different name or location.
FICHEIRO_DADOS = Path('2018- 2024 Luis Silvano - Doctoral Thesis Dataset.xlsx')

# 2) General filter: observations collected from 2018 through 2024 (inclusive).
ANO_INICIAL = 2018
ANO_FINAL = 2024

# 3) Temporal protocol: training -> validation -> testing.
ANOS_TREINO = [2018, 2019, 2020, 2021, 2022]
ANO_VALIDACAO = 2023
ANO_TESTE = 2024

# 4) Select 'temporal' OR 'hoteis_novos'.
PROTOCOLO = 'temporal'

# 5) For 'hoteis_novos', split by hotel (approximately 60/20/20).
FRACAO_TESTE_HOTEIS = 0.20
FRACAO_VALIDACAO_HOTEIS = 0.20

# 6) Initial test: models that run efficiently on a CPU.
#    To evaluate newer architectures, add the names listed in Section 8.
MODELOS_A_EXECUTAR = [
    'dummy', 'ridge', 'random_forest',
    'xgboost', 'lightgbm', 'catboost',
]

# 7) For new PyTorch models, choose 'cpu' or 'cuda' (NVIDIA GPU).
DISPOSITIVO = 'cpu'

# 8) Use year as a feature (can be disabled for an ablation study).
USAR_ANO_COMO_ATRIBUTO = True

# 9) Output directory; each protocol saves separate files.
PASTA_RESULTADOS = Path('resultados_kbs')

assert PROTOCOLO in {'temporal', 'hoteis_novos'}
assert len(set(ANOS_TREINO + [ANO_VALIDACAO, ANO_TESTE])) == len(ANOS_TREINO) + 2
assert min(ANOS_TREINO) >= ANO_INICIAL and ANO_TESTE <= ANO_FINAL
print('Configuration loaded. Protocol:', PROTOCOLO)
print('Years:', ANO_INICIAL, 'a', ANO_FINAL)
print('Selected models:', ', '.join(MODELOS_A_EXECUTAR))

## 2. Defining the Study Variables

The list below comes from the original notebook, with one important correction: **`ht_SurvialYearlog` is excluded from the main predictive model**. According to the original description, the survival of certain hotels is calculated using information available through 2024, which could reveal future information when predicting 2023/2024.

- `f_labprodct`: labour productivity, defined in the original notebook as sales / number of employees.
- `ht_capacity`: hotel capacity.
- `ht_log_number_of_accommodation_units`: logarithm of the number of accommodation units.
- `ht_location_beach1`, `ht_location_castle1`, `ht_location_church1`: distance to the nearest beach, castle, and church.
- `ht_stars_classification`: star rating.
- `ht_Restaurants&CoffesL800m`: indicator related to restaurants/cafés within an 800 m radius.
- `f_year`: observation year; known in advance, although we can test a model without using it as a feature.

**Check before publication:** Confirm how and when each attribute was measured. A model should only be described as *forecasting* if all predictors are available at prediction time. Filtering by year alone does not guarantee this.


In [ ]:
ID_HOTEL = 'ht_id'
COL_ANO = 'f_year'
COL_ALVO = 'f_labprodct'
COL_CADEIA = 'ht_hotel_chain_d'

ATRIBUTOS_HOTEL = [
    'ht_capacity',
    'ht_log_number_of_accommodation_units',
    'ht_location_beach1',
    'ht_location_castle1',
    'ht_location_church1',
    'ht_stars_classification',
    'ht_Restaurants&CoffesL800m',
]
ATRIBUTOS = ATRIBUTOS_HOTEL + ([COL_ANO] if USAR_ANO_COMO_ATRIBUTO else [])

print('Target:', COL_ALVO)
print('Number of features:', len(ATRIBUTOS))
print('Features:', ATRIBUTOS)

## 3. Loading the File and **Explicitly Filtering Years**

The order matters: **load → select independent hotels only → filter 2018–2024 → drop rows without an identifier, year, or target**. We do not automatically remove observations with missing features: these will be handled *after splitting the data*, using only the training set.

The original notebook read the Excel file from a Windows-specific path (`C:\Users\...`). Here, the default is a file located in the same folder as the notebook, making it easier to share across computers.


In [ ]:
if not FICHEIRO_DADOS.exists():
    raise FileNotFoundError(
        f"Could not find '{FICHEIRO_DADOS}'.\n"
        "Place the Excel file in the same folder as this notebook OR change the variable "
        "FICHEIRO_DADOS in Section 1. The original Excel file was not included."
    )

if FICHEIRO_DADOS.suffix.lower() in ('.xlsx', '.xls'):
    dados_brutos = pd.read_excel(FICHEIRO_DADOS)
elif FICHEIRO_DADOS.suffix.lower() == '.csv':
    dados_brutos = pd.read_csv(FICHEIRO_DADOS)
else:
    raise ValueError('Unrecognized format: use an .xlsx, .xls, or .csv file')

print('File opened:', FICHEIRO_DADOS)
print('Original dimensions (rows, columns):', dados_brutos.shape)
display(dados_brutos.head(3))

In [ ]:
colunas_necessarias = [ID_HOTEL, COL_ANO, COL_ALVO, COL_CADEIA] + ATRIBUTOS_HOTEL
em_falta = sorted(set(colunas_necessarias) - set(dados_brutos.columns))
if em_falta:
    raise ValueError(
        'The following columns from the original notebook were not found in the Excel file:\n'
        + '\n'.join(em_falta)
    )

# 1. Select only the required columns.
painel = dados_brutos[colunas_necessarias].copy()

# 2. Ensure the hotel-chain indicator and year are numeric.
painel[COL_CADEIA] = pd.to_numeric(painel[COL_CADEIA], errors='coerce')
painel[COL_ANO] = pd.to_numeric(painel[COL_ANO], errors='coerce')

# 3. Independent hotels: chain indicator = 0.
painel = painel.loc[painel[COL_CADEIA] == 0].copy()

# 4. EXPLICIT YEAR FILTER: 2018 through 2024, inclusive.
painel = painel.loc[painel[COL_ANO].between(ANO_INICIAL, ANO_FINAL)].copy()

# 5. Observations without an ID, year, or productivity cannot be modelled.
painel = painel.dropna(subset=[ID_HOTEL, COL_ANO, COL_ALVO]).copy()

# 6. Convert target and features to numbers; treat infinities as missing values.
for coluna in [COL_ALVO] + ATRIBUTOS_HOTEL:
    painel[coluna] = pd.to_numeric(painel[coluna], errors='coerce')
painel = painel.replace([np.inf, -np.inf], np.nan)
painel = painel.dropna(subset=[ID_HOTEL, COL_ANO, COL_ALVO]).copy()
painel[COL_ANO] = painel[COL_ANO].astype(int)
painel[ID_HOTEL] = painel[ID_HOTEL].astype(str)

# 7. One unique record per hotel-year; otherwise review the Excel merging process.
duplicados = painel.duplicated(subset=[ID_HOTEL, COL_ANO], keep=False)
if duplicados.any():
    display(painel.loc[duplicados, [ID_HOTEL, COL_ANO]].head(10))
    raise ValueError(
        f'Found {int(duplicados.sum())} rows with duplicate hotel/year identifiers. '
        'Review the source data; do not remove records automatically.'
    )

painel = painel.reset_index(drop=True)
if painel.empty:
    raise ValueError('No valid observations remain after filtering.')

print('After filtering:', painel.shape[0], 'observations')
print('Distinct independent hotels:', painel[ID_HOTEL].nunique())
print('Years included:', sorted(painel[COL_ANO].unique()))
assert painel[COL_ANO].between(ANO_INICIAL, ANO_FINAL).all()
display(painel.groupby(COL_ANO).agg(
    observacoes=(COL_ALVO, 'size'), hoteis=(ID_HOTEL, 'nunique')
).rename_axis('ano'))

## 4. Splitting the Data (*Train / Validation / Test*)

### Temporal protocol (main analysis)

```text
2018  2019  2020  2021  2022   |   2023   |   2024
[           TRAINING          ] | [VALID.] | [TEST]
```

**Note:** The same hotel may appear in training (2020), validation (2023), and testing (2024). This is permitted under this protocol; the key requirement is that no information from 2024 is used to fit the preprocessing steps or models.

### Unseen-hotel protocol (supplementary analysis)

Here, we perform a random split **by `ht_id`**, not by row. Therefore, all observations from a given hotel belong to only one set. All three sets may contain observations from any year between 2018 and 2024. This protocol answers a different question and should not be presented as a temporal forecasting test.


In [ ]:
def create_data_splits(df, protocolo, seed=42):
    if protocolo == 'temporal':
        treino = df.loc[df[COL_ANO].isin(ANOS_TREINO)].copy()
        validacao = df.loc[df[COL_ANO] == ANO_VALIDACAO].copy()
        teste = df.loc[df[COL_ANO] == ANO_TESTE].copy()
    elif protocolo == 'hoteis_novos':
        separador_teste = GroupShuffleSplit(
            n_splits=1, test_size=FRACAO_TESTE_HOTEIS, random_state=seed
        )
        id_treino_val, id_teste = next(
            separador_teste.split(df, groups=df[ID_HOTEL])
        )
        treino_val = df.iloc[id_treino_val].copy()
        teste = df.iloc[id_teste].copy()
        fracao_val_sobre_restantes = (
            FRACAO_VALIDACAO_HOTEIS / (1 - FRACAO_TESTE_HOTEIS)
        )
        separador_val = GroupShuffleSplit(
            n_splits=1, test_size=fracao_val_sobre_restantes,
            random_state=seed + 1
        )
        id_treino, id_val = next(
            separador_val.split(treino_val, groups=treino_val[ID_HOTEL])
        )
        treino = treino_val.iloc[id_treino].copy()
        validacao = treino_val.iloc[id_val].copy()
    else:
        raise ValueError('Unknown protocol')

    for nome, parte in [('treino', treino), ('validacao', validacao), ('teste', teste)]:
        if parte.empty:
            raise ValueError(f'The {nome} set is empty; review the years or filtering criteria.')
    return (treino.reset_index(drop=True),
            validacao.reset_index(drop=True),
            teste.reset_index(drop=True))


treino, validacao, teste = create_data_splits(painel, PROTOCOLO, SEED)

resumo = pd.DataFrame([
    {'conjunto': nome, 'observacoes': len(df), 'hoteis': df[ID_HOTEL].nunique(),
     'anos': ', '.join(map(str, sorted(df[COL_ANO].unique())))}
    for nome, df in [('treino', treino), ('validacao', validacao), ('teste', teste)]
])
display(resumo)

# Audit: always verify the conditions defining each protocol.
if PROTOCOLO == 'temporal':
    assert set(treino[COL_ANO]).issubset(set(ANOS_TREINO))
    assert set(validacao[COL_ANO]) == {ANO_VALIDACAO}
    assert set(teste[COL_ANO]) == {ANO_TESTE}
    print('OK: training covers 2018-2022 only; validation is 2023; testing is 2024.')
else:
    a, b, c = [set(df[ID_HOTEL]) for df in (treino, validacao, teste)]
    assert not(a & b or a & c or b & c)
    print('OK: no hotel appears in more than one of the training, validation, or test sets.')

mapa_divisao = pd.concat([
    df.assign(conjunto=nome)[[COL_ANO, 'conjunto']]
    for nome, df in [('treino', treino), ('validacao', validacao), ('teste', teste)]
], ignore_index=True)
print('Distribution by year and split:')
display(pd.crosstab(mapa_divisao[COL_ANO], mapa_divisao['conjunto']))

## 5. Training Data Diagnostics

To avoid unconsciously adapting models to the final test set, the quantitative target analysis below uses **training data only**. If some hotels have extremely high productivity, we will not automatically remove them: doing so would require a predefined substantive rule and sensitivity analysis.


In [ ]:
print('LABOUR productivity statistics (training set only):')
display(treino[COL_ALVO].describe().to_frame(name='produtividade'))

print('Missing values in explanatory features (training set only):')
display(
    (treino[ATRIBUTOS].isna().mean() * 100)
    .sort_values(ascending=False)
    .round(2)
    .rename('percentagem_em_falta')
    .to_frame()
)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(treino[COL_ALVO], bins=30)
axes[0].set_title('Productivity in the training set')
axes[0].set_xlabel(COL_ALVO)
axes[0].set_ylabel('Observations')
mediana_por_ano = treino.groupby(COL_ANO)[COL_ALVO].median()
axes[1].plot(mediana_por_ano.index, mediana_por_ano.values, marker='o')
axes[1].set_title('Median by year (training only)')
axes[1].set_xlabel('Year')
axes[1].set_ylabel('Median productivity')
plt.tight_layout()
plt.show()

## 6. Handling Missing Values **Without Data Leakage**

Some attributes may be missing for some hotels. A simple approach is to replace missing values with the **median calculated on the training set**. The *same* median is then applied to validation and test data.

**Example:** If the median capacity in the training data is 50 beds, use 50 to impute missing capacities across all three sets; never use the test-set median to impute the training set.

Scaling for Ridge regression is also contained within its own pipeline and fitted only on the training data. The hotel ID **is not a predictor**.


In [ ]:
atributos_usados = [c for c in ATRIBUTOS if treino[c].notna().any()]
atributos_excluidos = sorted(set(ATRIBUTOS) - set(atributos_usados))
if atributos_excluidos:
    warnings.warn(
        'No non-missing training values; excluded from this experiment: '
        + ', '.join(atributos_excluidos)
    )
if not atributos_usados:
    raise ValueError('No informative features remain in the training data.')

imputador = SimpleImputer(strategy='median')

X_treino = pd.DataFrame(
    imputador.fit_transform(treino[atributos_usados]),
    columns=atributos_usados, index=treino.index,
)
X_validacao = pd.DataFrame(
    imputador.transform(validacao[atributos_usados]),
    columns=atributos_usados, index=validacao.index,
)
X_teste = pd.DataFrame(
    imputador.transform(teste[atributos_usados]),
    columns=atributos_usados, index=teste.index,
)

y_treino = treino[COL_ALVO].to_numpy(dtype=float)
y_validacao = validacao[COL_ALVO].to_numpy(dtype=float)
y_teste = teste[COL_ALVO].to_numpy(dtype=float)

assert all(np.isfinite(x.to_numpy()).all() for x in (X_treino, X_validacao, X_teste))
assert all(np.isfinite(y).all() for y in (y_treino, y_validacao, y_teste))
print('OK. Features:', atributos_usados)
print('Shapes (training, validation, test):', X_treino.shape,
      X_validacao.shape, X_teste.shape)

## 7. Exploratory Analysis of Associations with the Target (Optional)

The original notebook used `f_classif`, which is intended for **classification**. Because `f_labprodct` is continuous, the corresponding univariate test is `f_regression`.

This analysis uses **training data only** and is intended to help understand the dataset; it **does not remove features** from the benchmark. Large F-statistics indicate meaningful *linear* associations, not causality, and do not capture nonlinear relationships. The table below should not replace an analysis of model interpretability.


In [ ]:
with warnings.catch_warnings():
    warnings.simplefilter('ignore', RuntimeWarning)
    F, p_valores = f_regression(X_treino, y_treino)

relacao_treino = pd.DataFrame({
    'atributo': atributos_usados,
    'estatistica_F': F,
    'valor_p': p_valores,
    'correlacao_spearman': [
        treino[c].corr(treino[COL_ALVO], method='spearman')
        for c in atributos_usados
    ],
}).sort_values('estatistica_F', ascending=False)

display(relacao_treino.round(4))

---
## 8. Available Regression Models

**First round (recommended):** `dummy`, `ridge`, `random_forest`, `xgboost`, `lightgbm`, `catboost`.

**More recent or widely cited tabular models (optional):**

| Name to add to `MODELOS_A_EXECUTAR` | Algorithm | Installation / note |
|---|---|---|
| `tabnet` | TabNet (AAAI 2021) | `pytorch-tabnet` |
| `ft_transformer` | FT-Transformer (NeurIPS 2021) | `pytabkit` |
| `mlp_plr` | MLP-PLR (NeurIPS 2022) | `pytabkit` |
| `tabr` | TabR (ICLR 2024) | `pytabkit` + FAISS |
| `realmlp` | RealMLP (NeurIPS 2024) | `pytabkit` |
| `tabm` | TabM (ICLR 2025) | `pytabkit` |
| `tabicl` | TabICLv2 (2026) | `tabicl`; weights downloaded upon first use |
| `tabpfn` | TabPFN (installed version) | `tabpfn`; check weight access/licence |
| `causilo` | Causilo (2026) | `causilo`; check PyTorch requirements/weight licence |
| `ebm` | Explainable Boosting Machine | `interpret` |

**Scientific note:** Different algorithms have different training requirements. At this stage, we use fixed initial configurations, *without systematic optimization*. These results constitute a **pilot experiment**, not a final KBS benchmark. For a publishable comparison, hyperparameter optimization budgets must be made comparable, and the testing protocol must be specified in advance.


In [ ]:
CATALOGO = {
    'dummy': ('sklearn', 'Baseline: always predict the training mean'),
    'ridge': ('sklearn', 'Regularized linear regression'),
    'random_forest': ('sklearn', 'Decision tree ensemble'),
    'xgboost': ('xgboost', 'Gradient boosting'),
    'lightgbm': ('lightgbm', 'Efficient gradient boosting'),
    'catboost': ('catboost', 'Gradient boosting'),
    'tabnet': ('pytorch_tabnet', 'Neural network with feature attention'),
    'ft_transformer': ('pytabkit', 'Transformer for tabular data'),
    'mlp_plr': ('pytabkit', 'Network with numeric feature representations'),
    'tabr': ('pytabkit', 'Neural network with similar-observation retrieval (requires FAISS)'),
    'realmlp': ('pytabkit', 'MLP with strong tabular configurations'),
    'tabm': ('pytabkit', 'Efficient tabular neural network ensemble'),
    'tabicl': ('tabicl', 'Foundation model tabular, in-context'),
    'tabpfn': ('tabpfn', 'Foundation model tabular'),
    'causilo': ('causilo', 'Foundation model tabular'),
    'ebm': ('interpret', 'Explainable additive model'),
}

nao_reconhecidos = sorted(set(MODELOS_A_EXECUTAR) - set(CATALOGO))
if nao_reconhecidos:
    raise ValueError(f'Unknown models: {nao_reconhecidos}')

def is_library_available(modulo):
    return True if modulo == 'sklearn' else importlib.util.find_spec(modulo) is not None

disponibilidade = pd.DataFrame([
    {'modelo': nome, 'biblioteca': CATALOGO[nome][0],
     'instalada': is_library_available(CATALOGO[nome][0]),
     'tipo': CATALOGO[nome][1]}
    for nome in MODELOS_A_EXECUTAR
])
display(disponibilidade)
if not disponibilidade['instalada'].all():
    print('Note: missing libraries will be reported as NOT RUN; no fictitious metrics will be generated.')

## 9. A Single Function for Constructing Each Model

You do not need to understand every implementation detail. The function takes the selected model name, constructs an object with its initial configuration, and returns it for training. This allows all models to be evaluated on the **same observations and features**.

Ridge regression includes scaling fitted on the training set; the other models receive data with missing values imputed using training medians.


In [ ]:
def build_model(nome, seed, dispositivo):
    if nome == 'dummy':
        return DummyRegressor(strategy='mean')
    if nome == 'ridge':
        return make_pipeline(StandardScaler(), Ridge(alpha=10.0))
    if nome == 'random_forest':
        return RandomForestRegressor(
            n_estimators=300, min_samples_leaf=3, random_state=seed, n_jobs=4
        )
    if nome == 'xgboost':
        from xgboost import XGBRegressor
        return XGBRegressor(
            n_estimators=500, learning_rate=0.05, max_depth=4,
            subsample=0.9, colsample_bytree=0.9, objective='reg:squarederror',
            random_state=seed, n_jobs=4,
        )
    if nome == 'lightgbm':
        from lightgbm import LGBMRegressor
        return LGBMRegressor(
            n_estimators=500, learning_rate=0.05, num_leaves=15,
            min_child_samples=25, random_state=seed, verbosity=-1, n_jobs=4,
        )
    if nome == 'catboost':
        from catboost import CatBoostRegressor
        return CatBoostRegressor(
            iterations=500, learning_rate=0.05, depth=5,
            loss_function='RMSE', random_seed=seed,
            verbose=False, thread_count=4,
        )
    if nome == 'tabnet':
        from pytorch_tabnet.tab_model import TabNetRegressor
        return TabNetRegressor(seed=seed, device_name=dispositivo, verbose=0)
    if nome in {'ft_transformer', 'mlp_plr', 'tabr', 'realmlp', 'tabm'}:
        from pytabkit import (
            FTT_D_Regressor, MLP_PLR_D_Regressor, TabR_S_D_Regressor,
            RealMLP_TD_Regressor, TabM_D_Regressor,
        )
        classes = {
            'ft_transformer': FTT_D_Regressor,
            'mlp_plr': MLP_PLR_D_Regressor,
            'tabr': TabR_S_D_Regressor,
            'realmlp': RealMLP_TD_Regressor,
            'tabm': TabM_D_Regressor,
        }
        return classes[nome](device=dispositivo, random_state=seed, n_cv=1)
    if nome == 'tabicl':
        from tabicl import TabICLRegressor
        return TabICLRegressor(device=dispositivo, random_state=seed, n_estimators=4)
    if nome == 'tabpfn':
        from tabpfn import TabPFNRegressor
        return TabPFNRegressor(device=dispositivo, random_state=seed, n_estimators=4)
    if nome == 'causilo':
        from causilo import CausiloRegressor
        return CausiloRegressor(device=dispositivo, random_state=seed, n_estimators=8)
    if nome == 'ebm':
        from interpret.glassbox import ExplainableBoostingRegressor
        return ExplainableBoostingRegressor(random_state=seed)
    raise ValueError(f'Unknown model: {nome}')


def train_and_predict(nome, modelo, Xtr, ytr, Xval, Xte):
    # TabNet requires NumPy arrays and y with shape (n_samples, 1).
    # No external early stopping, to keep the primary validation independent.
    if nome == 'tabnet':
        xtr = Xtr.to_numpy(dtype=np.float32)
        xval = Xval.to_numpy(dtype=np.float32)
        xte = Xte.to_numpy(dtype=np.float32)
        modelo.fit(
            xtr, ytr.reshape(-1, 1).astype(np.float32),
            max_epochs=60, patience=0,
            batch_size=256, virtual_batch_size=128,
        )
        return np.asarray(modelo.predict(xval)).reshape(-1), np.asarray(modelo.predict(xte)).reshape(-1)
    modelo.fit(Xtr, ytr)
    return np.asarray(modelo.predict(Xval)).reshape(-1), np.asarray(modelo.predict(Xte)).reshape(-1)


def calculate_metrics(y_real, y_previsto):
    return {
        'MAE': float(mean_absolute_error(y_real, y_previsto)),
        'RMSE': float(np.sqrt(mean_squared_error(y_real, y_previsto))),
        'R2': float(r2_score(y_real, y_previsto)),
    }

## 10. Running the Comparison

This is the main stage. For each model, the program:

1. Constructs the model and fits it **only on the training set**.
2. Predicts validation and test outcomes **without adjusting parameters using the test set**.
3. Computes MAE, RMSE, and R².
4. Stores hotel- and year-level predictions.
5. Explicitly logs execution errors (e.g., a missing library).

**Models are ranked by VALIDATION error, not TEST error.** Repeatedly checking the test results and modifying the code to improve them would constitute data leakage. For scientific submission, we should freeze all choices and evaluate the final test set only once.


In [ ]:
resultados_lista = []
falhas_lista = []
modelos_treinados = {}
previsoes_teste = teste[[ID_HOTEL, COL_ANO, COL_ALVO]].copy()
previsoes_validacao = validacao[[ID_HOTEL, COL_ANO, COL_ALVO]].copy()

for nome in MODELOS_A_EXECUTAR:
    modulo = CATALOGO[nome][0]
    if not is_library_available(modulo):
        msg = f"Library '{modulo}' is not installed. See the 'Installation' section."
        print(f'[NOT RUN] {nome}: {msg}')
        falhas_lista.append({'modelo': nome, 'motivo': msg})
        continue
    print(f'\nRunning: {nome} ...')
    try:
        modelo = build_model(nome, SEED, DISPOSITIVO)
        inicio = time.perf_counter()
        prev_val, prev_teste = train_and_predict(
            nome, modelo, X_treino, y_treino, X_validacao, X_teste
        )
        segundos = time.perf_counter() - inicio

        if len(prev_val) != len(y_validacao) or len(prev_teste) != len(y_teste):
            raise ValueError('The number of predictions does not match the number of observations.')
        if not (np.isfinite(prev_val).all() and np.isfinite(prev_teste).all()):
            raise ValueError('The model returned missing or infinite predictions.')

        res_val = calculate_metrics(y_validacao, prev_val)
        res_teste = calculate_metrics(y_teste, prev_teste)
        resultados_lista.append({
            'modelo': nome, 'segundos': round(segundos, 2),
            'val_MAE': res_val['MAE'], 'val_RMSE': res_val['RMSE'],
            'val_R2': res_val['R2'],
            'teste_MAE': res_teste['MAE'], 'teste_RMSE': res_teste['RMSE'],
            'teste_R2': res_teste['R2'],
        })
        previsoes_teste['previsao_' + nome] = prev_teste
        previsoes_validacao['previsao_' + nome] = prev_val
        modelos_treinados[nome] = modelo
        print(f"  Completed | validation RMSE={res_val['RMSE']:.4f} "
              f"| test RMSE={res_teste['RMSE']:.4f} | {segundos:.1f} s")
    except Exception as exc:
        msg = f'{type(exc).__name__}: {exc}'
        falhas_lista.append({'modelo': nome, 'motivo': msg})
        print(f'  [FAILED] {nome}: {msg}')

if not resultados_lista:
    raise RuntimeError('No model could be run successfully; see the errors above.')

resultados = pd.DataFrame(resultados_lista).sort_values('val_RMSE').reset_index(drop=True)
falhas = pd.DataFrame(falhas_lista, columns=['modelo', 'motivo'])

print('\nRESULTS (ranked by validation RMSE; lower is better)')
display(resultados.round(4))
if not falhas.empty:
    print('Models not run:')
    display(falhas)

## 11. How Should the Tables and Figures Be Interpreted?

- **MAE**: mean absolute error; measured in the same units as productivity.
- **RMSE**: similar to MAE, but more heavily penalizes large prediction errors.
- **R²**: may be negative. Negative values indicate worse performance than the mean-based reference on that set.
- **`dummy` baseline**: predicts a constant. If much more sophisticated neural networks cannot outperform this baseline, we should investigate data quality and the predictability of the task.

The next figure shows **validation error**, which is used to rank the models. Test performance is reported in the table, but **must not determine the winner during development**.


In [ ]:
fig, ax = plt.subplots(figsize=(10, max(3, 0.50 * len(resultados) + 1)))
ax.barh(resultados['modelo'][::-1], resultados['val_RMSE'][::-1])
ax.set_xlabel('Validation RMSE (lower is better)')
ax.set_title(f'Regression comparison - {PROTOCOLO} protocol')
plt.tight_layout()
plt.show()

modelo_escolhido_validacao = str(resultados.iloc[0]['modelo'])
print('Best VALIDATION RMSE:', modelo_escolhido_validacao)
print('This does NOT mean we already have a final publishable model.')

fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(
    y_teste, previsoes_teste['previsao_' + modelo_escolhido_validacao],
    s=16, alpha=0.55,
)
limite_min = min(float(np.min(y_teste)),
                 float(previsoes_teste['previsao_' + modelo_escolhido_validacao].min()))
limite_max = max(float(np.max(y_teste)),
                 float(previsoes_teste['previsao_' + modelo_escolhido_validacao].max()))
ax.plot([limite_min, limite_max], [limite_min, limite_max], '--', linewidth=1)
ax.set_xlabel('Observed productivity (test set)')
ax.set_ylabel('Predicted productivity')
ax.set_title(f'Test: {modelo_escolhido_validacao}')
plt.tight_layout()
plt.show()

## 12. Initial Feature Interpretation (Conventional Models Only)

We can measure how much error increases when a feature is randomly shuffled in the **validation** set. This is called *permutation importance*: if shuffling distance to the beach increases the error, that feature was useful to the model's predictions.

**Caution:** Feature importance does not necessarily represent a causal effect. Correlated predictors may exchange importance. This section is exploratory; for the paper, we will need to examine the stability of explanations more thoroughly using SHAP/ALE/EBM or other model-appropriate methods.


In [ ]:
from sklearn.inspection import permutation_importance

modelos_compativeis_permutacao = {
    'dummy', 'ridge', 'random_forest', 'xgboost', 'lightgbm', 'catboost', 'ebm'
}
modelo_para_explicar = next(
    (m for m in resultados['modelo'] if m in modelos_compativeis_permutacao), None
)

importancias = None
if modelo_para_explicar is None:
    print('No conventional model compatible with this explanation was trained.')
else:
    print('Computing feature importance for model:', modelo_para_explicar)
    perm = permutation_importance(
        modelos_treinados[modelo_para_explicar],
        X_validacao, y_validacao,
        n_repeats=5, random_state=SEED, scoring='neg_mean_absolute_error',
    )
    importancias = pd.DataFrame({
        'atributo': atributos_usados,
        'aumento_medio_MAE': perm.importances_mean,
        'desvio_padrao': perm.importances_std,
    }).sort_values('aumento_medio_MAE', ascending=False)
    display(importancias.round(4))
    ax = importancias.iloc[::-1].plot.barh(
        x='atributo', y='aumento_medio_MAE', legend=False, figsize=(9, 4)
    )
    ax.set_xlabel('MAE increase after feature permutation (validation)')
    ax.set_title(f'Exploratory feature importance - {modelo_para_explicar}')
    plt.tight_layout()
    plt.show()

## 13. Automatic Export

Results are stored in `resultados_kbs/`. File names include the protocol so temporal results are not mixed with unseen-hotel results.

| File | Contents |
|---|---|
| `metricas_[protocolo].csv` | Validation and test MAE, RMSE, R²; execution time. |
| `previsoes_teste_[protocolo].csv` | Observed and predicted values, identified by hotel and year. |
| `previsoes_validacao_[protocolo].csv` | Same information for validation. |
| `falhas_[protocolo].csv` | Missing libraries or models that failed. |
| `contagens_[protocolo].csv` | Observation and hotel counts in each set. |
| `configuracao_[protocolo].json` | Years, features, seed, protocol, and library versions. |
| `importancias_[protocolo].csv` | Exploratory feature importances, if computed. |

**Never present results for models listed in `falhas_...csv` as if those models had successfully run.**


In [ ]:
PASTA_RESULTADOS.mkdir(parents=True, exist_ok=True)

ficheiros = {
    'metricas': PASTA_RESULTADOS / f'metricas_{PROTOCOLO}.csv',
    'previsoes_teste': PASTA_RESULTADOS / f'previsoes_teste_{PROTOCOLO}.csv',
    'previsoes_validacao': PASTA_RESULTADOS / f'previsoes_validacao_{PROTOCOLO}.csv',
    'falhas': PASTA_RESULTADOS / f'falhas_{PROTOCOLO}.csv',
    'contagens': PASTA_RESULTADOS / f'contagens_{PROTOCOLO}.csv',
    'configuracao': PASTA_RESULTADOS / f'configuracao_{PROTOCOLO}.json',
}
resultados.to_csv(ficheiros['metricas'], index=False)
previsoes_teste.to_csv(ficheiros['previsoes_teste'], index=False)
previsoes_validacao.to_csv(ficheiros['previsoes_validacao'], index=False)
falhas.to_csv(ficheiros['falhas'], index=False)
resumo.to_csv(ficheiros['contagens'], index=False)

if importancias is not None:
    ficheiros['importancias'] = PASTA_RESULTADOS / f'importancias_{PROTOCOLO}.csv'
    importancias.to_csv(ficheiros['importancias'], index=False)

versoes = {}
for distribuicao in ['python', 'numpy', 'pandas', 'scikit-learn', 'xgboost',
                    'lightgbm', 'catboost', 'pytabkit', 'pytorch-tabnet',
                    'tabicl', 'tabpfn', 'causilo', 'interpret']:
    if distribuicao == 'python':
        versoes[distribuicao] = platform.python_version()
        continue
    try:
        versoes[distribuicao] = importlib.metadata.version(distribuicao)
    except importlib.metadata.PackageNotFoundError:
        versoes[distribuicao] = None

configuracao_registada = {
    'ficheiro_entrada': str(FICHEIRO_DADOS),
    'protocolo': PROTOCOLO,
    'filtro_anos': [ANO_INICIAL, ANO_FINAL],
    'anos_treino_temporal': ANOS_TREINO,
    'ano_validacao_temporal': ANO_VALIDACAO,
    'ano_teste_temporal': ANO_TESTE,
    'semente': SEED,
    'dispositivo': DISPOSITIVO,
    'alvo': COL_ALVO,
    'atributos': atributos_usados,
    'atributos_removidos_por_ausencia_total_no_treino': atributos_excluidos,
    'atributos_excluidos_por_fuga_temporal': ['ht_SurvialYearlog'],
    'modelos_pedidos': MODELOS_A_EXECUTAR,
    'modelos_executados': resultados['modelo'].tolist(),
    'falhas': falhas_lista,
    'contagens': resumo.to_dict(orient='records'),
    'versoes': versoes,
    'nota': ('Pilot study without HPO. Interpret temporal and hoteis_novos '
             'as different questions. Confirm temporal availability of features.'),
}
ficheiros['configuracao'].write_text(
    json.dumps(configuracao_registada, ensure_ascii=False, indent=2),
    encoding='utf-8',
)

print('Saved files:')
for tipo, caminho in ficheiros.items():
    print(f'  {tipo}: {caminho.resolve()}')

---
## 14. What Remains Before Submission to KBS?

This notebook provides a **reproducible experimental foundation**, but does not replace the following scientific steps:

1. **Temporal audit of features:** Confirm that each variable was known in the year when it would be used, including any subsequent revisions of hotel information. `ht_SurvialYearlog` remains excluded from the main temporal experiment.
2. **Define the objective:** Predict 2024 from earlier years, predict for unseen hotels, or estimate productivity within the same year (*nowcasting*). Each objective requires a different protocol.
3. **Fair optimization:** Establish comparable hyperparameter budgets, control random seeds, and avoid test-set-driven development decisions.
4. **Check outliers and skewness:** Determine whether extreme productivity values dominate RMSE. If the target is transformed, fit that transformation only on training data and report metrics on the original scale.
5. **Robustness:** Repeat hotel-based partitions, use year-based/rolling-origin tests when sample size permits, and report uncertainty and between-hotel variability.
6. **Explainability and knowledge discovery:** Investigate the stability of associations with location, hotel size, and star rating using SHAP/ALE/EBM, without incorrectly inferring causation.
7. **Reproducibility:** Save library versions, final hyperparameters, data extraction date, and a data dictionary. Check permissions and weight-licence conditions for foundation models.

### Further Reading / Implementations to Check

- [TabNet](https://github.com/dreamquark-ai/tabnet)
- [PyTabKit: FT-Transformer, MLP-PLR, TabR, RealMLP, TabM](https://github.com/dholzmueller/pytabkit)
- [TabICL](https://tabicl.readthedocs.io/en/latest/)
- [TabPFN](https://github.com/PriorLabs/TabPFN)
- [Causilo](https://pypi.org/project/causilo/)

### Frequently Asked Questions

**"Aren't we filtering the years?"** Yes, we are: Section 3 filters 2018–2024, and Section 4 separates the subsets according to the selected protocol.

**"Why does the same hotel appear in two subsets?"** This is normal in a temporal test (different years). To test on previously unseen hotels, choose `hoteis_novos`.

**"Why did a model fail?"** Possible reasons include a missing package, pretrained weights, licence, GPU, memory, or a dependency such as FAISS. Check `falhas_[protocolo].csv`; missing results are never fabricated.

**"Why don't we use all 200 features in the Excel file?"** The original study selected specific structural and locational features. Expanding the feature set requires justification of temporal availability and avoidance of variables derived from the target (e.g., same-year sales/employee counts).

**"Where is the old code?"** The original file is preserved separately. This notebook is a **clean, unified reconstruction of the core workflow**, not a reproduction of every descriptive table from the original.
